# Training Neural Networks

The previous notebooks built intuition for how perceptrons and neural networks are *structured*. This notebook covers how they are *trained* — specifically:

1. How the vanilla neural network produces outputs (including **softmax** for multi-class classification)
2. How **backpropagation** computes gradients through the network
3. The difference between **batch gradient descent** and **stochastic gradient descent**
4. Common **practical issues** and how to address them

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.datasets import make_moons
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

## The Vanilla Neural Network

A single-hidden-layer ("vanilla") neural network has three layers:

- **Input layer:** $p$ features $X_1, \ldots, X_p$
- **Hidden layer:** $M$ nodes $Z_1, \ldots, Z_M$
- **Output layer:** $K$ outputs $Y_1, \ldots, Y_K$

Each hidden node computes:
$$Z_m = \sigma(\alpha_{0m} + \alpha_m^T X), \quad m = 1, \ldots, M$$

where $\sigma$ is an activation function and $\alpha_m$ is the vector of weights connecting the input layer to node $m$.

Each output is then a linear combination of the hidden nodes:
$$T_k = \beta_{0k} + \beta_k^T Z, \quad k = 1, \ldots, K$$

The final prediction applies an **output function** $g_k$ to $T$:
$$\hat{Y}_k = f_k(X) = g_k(T)$$

For **regression**, the identity function is used: $g_k(T) = T_k$.

For **classification**, we need outputs that sum to 1 across classes — that's where softmax comes in.

## Softmax Output for Classification

When we have $K$ classes, we need each output $\hat{Y}_k$ to represent the probability that an observation belongs to class $k$. That means the outputs must:
- All be between 0 and 1
- Sum to 1 across all $K$ classes

The **softmax** function achieves this:
$$g_k(T) = \frac{e^{T_k}}{\sum_{l=1}^{K} e^{T_l}}$$

The final classifier picks the class with the highest probability:
$$G(x) = \arg\max_k f_k(x)$$

For binary classification ($K = 2$), softmax reduces to the sigmoid function.

In [ ]:
def softmax(T):
    e = np.exp(T - np.max(T))  # subtract max for numerical stability
    return e / e.sum()

# Example: 4-class network with raw output scores T
T = np.array([2.0, 1.0, 0.1, -0.5])
probs = softmax(T)

print('Raw output scores T:', T)
print('Softmax probabilities:', np.round(probs, 4))
print('Sum of probabilities:', probs.sum())
print('Predicted class:', np.argmax(probs))

## Loss Functions

Training a neural network means finding weights $\theta = \{\alpha_{jm}, \beta_{mk}\}$ that minimize a **loss function** $R(\theta)$ over the training data.

For **regression**, we use sum-of-squared errors:
$$R(\theta) = \sum_{k=1}^{K} \sum_{i=1}^{N} \left(y_{ik} - f_k(x_i)\right)^2$$

For **classification**, we use **cross-entropy** (also called log loss):
$$R(\theta) = -\sum_{k=1}^{K} \sum_{i=1}^{N} y_{ik} \log f_k(x_i)$$

Cross-entropy penalizes confident wrong predictions heavily — if the model assigns near-zero probability to the correct class, the loss goes to infinity.

## Backpropagation

To minimize $R(\theta)$ we need to compute the gradient of the loss with respect to every weight in the network. **Backpropagation** is a two-pass algorithm that does this efficiently.

### Forward Pass
Fix the current weights and compute predicted values $\hat{f}_k(x_i)$ from input to output, layer by layer.

### Backward Pass
Starting from the output layer, propagate the **error** backwards through the network.

For each output node, compute the **output error**:
$$\delta_{ki} = -2(y_{ik} - f_k(x_i)) \cdot g_k'(\beta_k^T z_i)$$

For each hidden node, compute the **hidden error** by summing how much each output node blamed this hidden node:
$$s_{mi} = \sigma'(\alpha_m^T x_i) \sum_{k=1}^{K} \beta_{km} \delta_{ki}$$

The partial derivatives of the loss with respect to the weights are then:
$$\frac{\partial R_i}{\partial \beta_{km}} = \delta_{ki} z_{mi} \qquad \frac{\partial R_i}{\partial \alpha_{ml}} = s_{mi} x_{il}$$

This is where the name "backpropagation" comes from: the output errors $\delta_{ki}$ flow *backwards* through the network to produce the hidden errors $s_{mi}$, which in turn tell us how to update the weights in the earlier layer.

The key requirement is that activation functions must be **differentiable** (or at least have a well-defined subgradient) — which is why the binary step function is not used in hidden layers of trainable networks.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(12, 6))
ax.set_xlim(0, 12); ax.set_ylim(0, 8)
ax.axis('off')
ax.set_title(
    'Backpropagation: Forward Pass computes predictions; Backward Pass propagates errors',
    fontsize=12, pad=12)

input_pos  = [(1.8, y) for y in [2.0, 4.0, 6.0]]
hidden_pos = [(5.5, y) for y in [1.5, 3.5, 5.5]]
output_pos = [(9.2, y) for y in [3.0, 5.0]]
R = 0.45

def draw_node(pos, label, facecolor):
    c = plt.Circle(pos, R, color=facecolor, zorder=4, ec='white', lw=1.5)
    ax.add_patch(c)
    ax.text(pos[0], pos[1], label, ha='center', va='center',
            fontsize=10, fontweight='bold', color='white', zorder=5)

def draw_edge(a, b, color, alpha, lw=1.2):
    dx, dy = b[0]-a[0], b[1]-a[1]
    L = (dx**2+dy**2)**0.5
    s = R/L
    ax.annotate('', xy=(b[0]-dx*s, b[1]-dy*s), xytext=(a[0]+dx*s, a[1]+dy*s),
                arrowprops=dict(arrowstyle='->', color=color, lw=lw, alpha=alpha), zorder=3)

for inp in input_pos:
    for hid in hidden_pos:
        draw_edge(inp, hid, '#2E86AB', 0.45)
for hid in hidden_pos:
    for out in output_pos:
        draw_edge(hid, out, '#2E86AB', 0.45)
for hid in hidden_pos:
    for out in output_pos:
        draw_edge(out, hid, '#C1292E', 0.35, lw=1.0)
for inp in input_pos:
    for hid in hidden_pos:
        draw_edge(hid, inp, '#C1292E', 0.35, lw=1.0)

for i, pos in enumerate(input_pos):  draw_node(pos, f'x{i+1}', '#2E86AB')
for i, pos in enumerate(hidden_pos): draw_node(pos, f'z{i+1}', '#6B4226')
for i, pos in enumerate(output_pos): draw_node(pos, f'y{i+1}', '#A23B72')

for x, label, color in [(1.8,'Input\nLayer','#2E86AB'),
                         (5.5,'Hidden\nLayer','#6B4226'),
                         (9.2,'Output\nLayer','#A23B72')]:
    ax.text(x, 7.3, label, ha='center', va='center',
            fontsize=10, fontweight='bold', color=color)

ax.text(3.7, 4.8, 'α weights', ha='center', fontsize=9, color='gray', style='italic')
ax.text(7.3, 4.8, 'β weights', ha='center', fontsize=9, color='gray', style='italic')

ax.annotate('', xy=(10.8, 6.9), xytext=(0.5, 6.9),
            arrowprops=dict(arrowstyle='->', color='#2E86AB', lw=2.5))
ax.text(5.5, 7.05, 'FORWARD: compute predictions  fk(x)', ha='center',
        fontsize=10, color='#2E86AB')

ax.annotate('', xy=(0.5, 0.4), xytext=(10.8, 0.4),
            arrowprops=dict(arrowstyle='->', color='#C1292E', lw=2.5))
ax.text(5.5, 0.15, 'BACKWARD: propagate errors  δ (output layer)  →  s (hidden layer)',
        ha='center', fontsize=10, color='#C1292E')

fwd = mpatches.Patch(color='#2E86AB', alpha=0.7, label='Forward pass')
bwd = mpatches.Patch(color='#C1292E', alpha=0.7, label='Backward pass (gradients)')
ax.legend(handles=[fwd, bwd], loc='lower right', fontsize=10)
plt.tight_layout()
plt.show()

## Gradient Descent: Batch vs. Stochastic

Once we have gradients, we update the weights by moving in the direction that reduces the loss:
$$\theta^{(r+1)} = \theta^{(r)} - \gamma_r \nabla R(\theta^{(r)})$$

where $\gamma_r$ is the **learning rate** at step $r$. There are two main strategies for how to compute this gradient:

### Batch Gradient Descent
Compute the gradient using **all $N$ training examples** before updating the weights:
$$\beta_{km}^{(r+1)} = \beta_{km}^{(r)} - \gamma_r \sum_{i=1}^{N} \frac{\partial R_i}{\partial \beta_{km}^{(r)}}$$

- Each update is expensive but uses the most accurate gradient estimate
- Learning rate $\gamma$ is typically constant
- One full pass through training data = one **epoch**

### Stochastic Gradient Descent (SGD)
Update the weights after **each individual training example**:
$$\beta_{km}^{(r+1)} = \beta_{km}^{(r)} - \gamma_r \frac{\partial R_i}{\partial \beta_{km}^{(r)}}$$

- Updates are noisy but much faster per epoch
- Training points are shuffled at the start of each epoch to prevent ordering effects
- Learning rate is decreased over time: $\gamma_r \to 0$ with $\sum_r \gamma_r = \infty$ and $\sum_r \gamma_r^2 < \infty$ — for example, $\gamma_r = 1/r$
- Handles very large datasets and can incorporate new examples as they arrive

**Mini-batch gradient descent** (the most common in practice) splits the difference: gradient is computed on a small random batch of examples (e.g. 32 or 128) per update.

In [ ]:
# Generate a dataset to demonstrate batch vs SGD training
X, y = make_moons(n_samples=500, noise=0.25, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train with batch gradient descent (lbfgs solver uses full-batch)
batch_model = MLPClassifier(hidden_layer_sizes=(20,), activation='tanh',
                             solver='lbfgs', max_iter=500, random_state=42)
batch_model.fit(X_train, y_train)

# Train with SGD
sgd_model = MLPClassifier(hidden_layer_sizes=(20,), activation='tanh',
                           solver='sgd', max_iter=500, random_state=42,
                           learning_rate_init=0.01)
sgd_model.fit(X_train, y_train)

print(f'Batch (lbfgs) test accuracy:  {accuracy_score(y_test, batch_model.predict(X_test)):.4f}')
print(f'SGD test accuracy:             {accuracy_score(y_test, sgd_model.predict(X_test)):.4f}')

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

X_cmp, y_cmp = make_moons(n_samples=200, noise=0.25, random_state=42)
X_cmp = StandardScaler().fit_transform(X_cmp)

configs = [
    ('Full-batch GD\n(batch_size = N)',    len(X_cmp), 'steelblue'),
    ('Mini-batch SGD\n(batch_size = 32)',  32,          'darkorange'),
    ('True SGD\n(batch_size = 1)',         1,            'crimson'),
]

fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)

for ax, (label, bs, color) in zip(axes, configs):
    m = MLPClassifier(
        hidden_layer_sizes=(20,), activation='tanh', solver='sgd',
        max_iter=300, random_state=42, learning_rate_init=0.01,
        batch_size=bs, tol=0, n_iter_no_change=10000, verbose=False
    )
    m.fit(X_cmp, y_cmp)
    ax.plot(m.loss_curve_, color=color, linewidth=1.5, alpha=0.9)
    ax.set_title(label, fontsize=10)
    ax.set_xlabel('Epoch')
    if ax is axes[0]:
        ax.set_ylabel('Training Loss')
    ax.grid(alpha=0.3)

fig.suptitle(
    'Same model, same data, same learning rate — only batch size differs.\n'
    'Full-batch: smooth but slow per update.  True SGD: noisy but can escape local minima faster.',
    fontsize=10
)
plt.tight_layout()
plt.show()

## Potential Issues

### 1. Starting Values and the Vanishing Gradient Problem

Weights are initialized randomly near 0, which starts the network in a nearly linear regime and allows it to grow more complex as training proceeds.

- Weights initialized to **exactly 0** give gradients of 0 — the algorithm never moves. This is the **vanishing gradient problem** in its simplest form.
- Sigmoid and TanH also cause vanishing gradients in deep networks because their derivatives approach 0 for large inputs. Errors propagated through many layers shrink to essentially nothing.
- **Large starting weights** lead to saturation and poor solutions.
- ReLU avoids this for positive inputs (derivative = 1), which is a major reason it became the default hidden-layer activation.

In [ ]:
# Illustrate vanishing gradient: sigmoid derivative shrinks rapidly away from 0
x = np.linspace(-6, 6, 400)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_deriv(x):
    s = sigmoid(x)
    return s * (1 - s)

def tanh_deriv(x):
    return 1 - np.tanh(x)**2

def relu_deriv(x):
    return np.where(x > 0, 1.0, 0.0)

fig, axes = plt.subplots(1, 3, figsize=(12, 3), sharey=True)
for ax, (fn, label, color) in zip(axes, [
    (sigmoid_deriv, "Sigmoid derivative", 'crimson'),
    (tanh_deriv,    "TanH derivative",    'mediumpurple'),
    (relu_deriv,    "ReLU derivative",    'darkorange'),
]):
    ax.plot(x, fn(x), color=color, linewidth=2)
    ax.set_title(label)
    ax.set_xlabel('x')
    ax.axhline(0, color='gray', linewidth=0.5)
axes[0].set_ylabel('Derivative value')
fig.suptitle('Activation Function Derivatives — gradient magnitude during backprop', y=1.02)
plt.tight_layout()
plt.show()

print("Sigmoid and TanH derivatives approach 0 for large |x|.")
print("ReLU derivative is 1 for all positive inputs — gradients don't shrink.")

### 2. Overfitting

Neural networks have many parameters and will overfit if not constrained. Two main remedies:

**Early stopping:** Stop training after a fixed number of epochs before the loss reaches its minimum. Because the network starts nearly linear and grows more complex over epochs, stopping early constrains complexity. A validation set is used to decide when to stop.

**Weight decay (regularization):** Add a penalty term to the loss and minimize $R(\theta) + \lambda J(\theta)$, where $\lambda$ controls the strength of regularization. Two common forms:
$$J(\theta) = \sum_{k,m} \beta_{km}^2 + \sum_{m,l} \alpha_{ml}^2 \qquad \text{(L2 / ridge)}$$
$$J(\theta) = \sum_{k,m} \frac{\beta_{km}^2}{1 + \beta_{km}^2} + \sum_{m,l} \frac{\alpha_{ml}^2}{1 + \alpha_{ml}^2} \qquad \text{(soft weight decay)}$$

In sklearn, the weight decay parameter is called `alpha`.

In [ ]:
# Demo: overfitting with too many epochs; early stopping as remedy
np.random.seed(0)
X_syn = np.random.normal(size=(2, 200))
a1 = np.array([[3], [3]])
a2 = np.array([[3], [-3]])

def sigmoid_fn(x):
    return 1 / (1 + np.exp(-x))

y_syn = (sigmoid_fn(a1.T @ X_syn) + (a2.T @ X_syn)**2 + 0.3 * np.random.normal(size=(1, 200)))[0]
X_syn = X_syn.T

X_tr, X_te, y_tr, y_te = train_test_split(X_syn, y_syn, test_size=0.5, random_state=1)

epoch_counts = list(range(10, 501, 10))
train_errors, test_errors = [], []

for ep in epoch_counts:
    m = MLPRegressor(hidden_layer_sizes=(10,), activation='tanh', solver='sgd',
                     max_iter=ep, random_state=1, learning_rate_init=0.01,
                     alpha=0, tol=1e-10, n_iter_no_change=ep+1)
    m.fit(X_tr, y_tr)
    train_errors.append(np.mean((m.predict(X_tr) - y_tr)**2))
    test_errors.append(np.mean((m.predict(X_te) - y_te)**2))

best_epoch = epoch_counts[np.argmin(test_errors)]

plt.figure(figsize=(8, 4))
plt.plot(epoch_counts, train_errors, label='Training error', color='steelblue')
plt.plot(epoch_counts, test_errors,  label='Test error',     color='crimson')
plt.axvline(best_epoch, color='gray', linestyle='--', label=f'Best epoch ({best_epoch})')
plt.xlabel('Epochs')
plt.ylabel('MSE')
plt.title('Overfitting: Training vs. Test Error by Epoch')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Test error is minimized at epoch {best_epoch}.')
print('Training past this point reduces training error but increases test error — overfitting.')

### 3. Input Scaling

Neural network weights are sensitive to the scale of inputs. If one feature ranges from 0–1 and another from 0–10,000, the network will assign very different weight magnitudes to each just to compensate for scale, making training unstable.

**Best practice:** Standardize inputs to mean 0 and standard deviation 1 before training.

In [ ]:
X_raw, y_cls = make_moons(n_samples=400, noise=0.2, random_state=0)

# Artificially inflate the second feature to create a scale mismatch
X_scaled_wrong = X_raw.copy()
X_scaled_wrong[:, 1] *= 1000

scaler = StandardScaler()
X_scaled_right = scaler.fit_transform(X_scaled_wrong)

def train_and_score(X_data, label):
    Xtr, Xte, ytr, yte = train_test_split(X_data, y_cls, test_size=0.25, random_state=1)
    m = MLPClassifier(hidden_layer_sizes=(10,), activation='tanh', solver='sgd',
                      max_iter=300, random_state=1, learning_rate_init=0.01)
    m.fit(Xtr, ytr)
    acc = accuracy_score(yte, m.predict(Xte))
    print(f'{label}: test accuracy = {acc:.4f}')

train_and_score(X_scaled_wrong, 'Without scaling')
train_and_score(X_scaled_right, 'With StandardScaler')

### 4. Model Structure

Choosing the number of hidden layers and nodes per layer:

- **Too few hidden units:** The network is too inflexible to capture the true relationship.
- **Too many hidden units:** The network can overfit, but this can be addressed with regularization — so it's generally better to err on the side of more units and regularize.
- More training examples and more input features justify more hidden nodes.
- Number of hidden layers varies by problem. For most tabular data, one or two layers is sufficient. Deeper architectures are mainly useful for images, text, and audio.

**Recommended workflow for choosing model structure:**

1. Split data 70 / 20 / 10 as **training / validation / test**. Hold out the 10% test set until the very end.
2. Propose several model structures (number of layers and nodes).
3. For each structure, use cross-validation on the 70% training set to find the best hyperparameters (epochs and $\lambda$).
4. Train each structure on the full 70% with its best hyperparameters.
5. Compare structures by evaluating each on the 20% validation set. Choose the best structure.
6. Retrain the best structure on 90% of the data (training + validation).
7. Report final out-of-sample error on the held-out 10% test set.

This separates the data used to *fit* models (training) from the data used to *compare* models (validation), preventing leakage from model selection into the final error estimate.

In [ ]:
# Interactive widget: explore how hidden layer size and regularization affect fit
X_demo, y_demo = make_moons(n_samples=300, noise=0.3, random_state=7)
scaler_demo = StandardScaler()
X_demo = scaler_demo.fit_transform(X_demo)

X_tr_d, X_te_d, y_tr_d, y_te_d = train_test_split(X_demo, y_demo, test_size=0.25, random_state=7)

hidden_slider  = widgets.IntSlider(value=10, min=1, max=50, step=1,
                                    description='Hidden nodes', style={'description_width': 'initial'})
alpha_slider   = widgets.FloatLogSlider(value=0.0001, base=10, min=-5, max=2, step=0.5,
                                         description='Regularization α', style={'description_width': 'initial'})
epochs_slider2 = widgets.IntSlider(value=200, min=10, max=1000, step=10,
                                    description='Epochs', style={'description_width': 'initial'})
run_btn        = widgets.Button(description='Train', button_style='primary')
out2           = widgets.Output()

def on_train(b):
    with out2:
        out2.clear_output(wait=True)
        model = MLPClassifier(
            hidden_layer_sizes=(hidden_slider.value,),
            activation='tanh', solver='sgd',
            max_iter=epochs_slider2.value,
            alpha=alpha_slider.value,
            random_state=1, learning_rate_init=0.05,
            tol=1e-10, n_iter_no_change=epochs_slider2.value + 1
        )
        model.fit(X_tr_d, y_tr_d)
        tr_acc = accuracy_score(y_tr_d, model.predict(X_tr_d))
        te_acc = accuracy_score(y_te_d, model.predict(X_te_d))

        # Decision boundary
        xx, yy = np.meshgrid(np.linspace(-3, 3, 200), np.linspace(-3, 3, 200))
        Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

        ax1.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
        ax1.scatter(X_tr_d[:, 0], X_tr_d[:, 1], c=y_tr_d, cmap='coolwarm', edgecolors='k', s=20)
        ax1.set_title(f'Decision Boundary\nTrain acc: {tr_acc:.3f} | Test acc: {te_acc:.3f}')
        ax1.set_xlabel('Feature 1')
        ax1.set_ylabel('Feature 2')

        ax2.plot(model.loss_curve_, color='steelblue')
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('Loss')
        ax2.set_title('Training Loss Curve')

        if tr_acc - te_acc > 0.05:
            ax1.set_title(ax1.get_title() + '  ⚠ possible overfit')

        plt.tight_layout()
        plt.show()

run_btn.on_click(on_train)
display(widgets.VBox([hidden_slider, alpha_slider, epochs_slider2, run_btn, out2]))
on_train(None)